# 01 - 画像カタログの作成

画像フォルダを再帰的にスキャンし、メタデータをDuckDBに保存する。

## 入力
- `data/hdd-images/` フォルダ（第1階層がカテゴリ）

## 出力
- `data/images.duckdb` (image_catalog テーブル)

In [1]:
import uuid
from pathlib import Path

import duckdb
from PIL import Image
from tqdm.notebook import tqdm

## 設定

In [2]:
# 画像フォルダのパス
IMAGE_ROOT = Path("../data/hdd-images")

# DuckDBファイルのパス
DB_PATH = Path("../data/images.duckdb")

# 対象の画像拡張子
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp"}

## DuckDBテーブル作成

In [3]:
# データベース接続
conn = duckdb.connect(str(DB_PATH))

# テーブル作成
conn.execute("""
    CREATE TABLE IF NOT EXISTS image_catalog (
        id VARCHAR PRIMARY KEY,
        file_path VARCHAR NOT NULL,
        relative_path VARCHAR,
        category VARCHAR,
        file_name VARCHAR,
        file_size INTEGER,
        width INTEGER,
        height INTEGER,
        format VARCHAR,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

print("テーブル作成完了")

テーブル作成完了


## 画像スキャン関数

In [4]:
def get_image_info(file_path: Path, image_root: Path) -> dict | None:
    """画像ファイルの情報を取得する"""
    try:
        # 相対パスを計算
        relative_path = file_path.relative_to(image_root)
        
        # 第1階層のフォルダ名をカテゴリとして取得
        parts = relative_path.parts
        category = parts[0] if len(parts) > 1 else "uncategorized"
        
        # ファイルサイズ
        file_size = file_path.stat().st_size
        
        # 画像情報を取得
        with Image.open(file_path) as img:
            width, height = img.size
            img_format = img.format
        
        return {
            "id": str(uuid.uuid4()),
            "file_path": str(file_path.resolve()),
            "relative_path": str(relative_path),
            "category": category,
            "file_name": file_path.name,
            "file_size": file_size,
            "width": width,
            "height": height,
            "format": img_format,
        }
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [5]:
def scan_images(image_root: Path, extensions: set[str]) -> list[Path]:
    """画像ファイルを再帰的にスキャン"""
    image_files = []
    for ext in extensions:
        image_files.extend(image_root.rglob(f"*{ext}"))
        image_files.extend(image_root.rglob(f"*{ext.upper()}"))
    
    # macOSのリソースフォーク (._で始まるファイル) を除外
    image_files = [f for f in image_files if not f.name.startswith("._")]
    
    return sorted(set(image_files))

## 画像スキャン実行

In [6]:
# 画像ファイルを検索
print(f"スキャン対象: {IMAGE_ROOT}")
image_files = scan_images(IMAGE_ROOT, IMAGE_EXTENSIONS)
print(f"見つかった画像ファイル数: {len(image_files)}")

スキャン対象: ../data/hdd-images
見つかった画像ファイル数: 378


## DuckDBへの保存

In [7]:
# 既存データをクリア（再実行時）
conn.execute("DELETE FROM image_catalog")

# 画像情報を取得してDBに保存
success_count = 0
error_count = 0

for file_path in tqdm(image_files, desc="画像を処理中"):
    info = get_image_info(file_path, IMAGE_ROOT)
    if info:
        conn.execute("""
            INSERT INTO image_catalog 
            (id, file_path, relative_path, category, file_name, file_size, width, height, format)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            info["id"],
            info["file_path"],
            info["relative_path"],
            info["category"],
            info["file_name"],
            info["file_size"],
            info["width"],
            info["height"],
            info["format"],
        ])
        success_count += 1
    else:
        error_count += 1

print(f"\n処理完了: 成功={success_count}, エラー={error_count}")

画像を処理中:   0%|          | 0/378 [00:00<?, ?it/s]


処理完了: 成功=378, エラー=0


## 結果確認

In [8]:
# 総件数
total = conn.execute("SELECT COUNT(*) FROM image_catalog").fetchone()[0]
print(f"総画像数: {total}")

総画像数: 378


In [9]:
# カテゴリ別件数
categories = conn.execute("""
    SELECT category, COUNT(*) as count 
    FROM image_catalog 
    GROUP BY category 
    ORDER BY count DESC
""").fetchall()

print("カテゴリ別件数:")
for cat, count in categories:
    print(f"  {cat}: {count}")

カテゴリ別件数:
  EuroPython2025: 129
  PyConJP2025: 77
  PyConJP2025-PreCampHiroshima: 57
  KashiwaVillagePark2026: 56
  TokyoNight202505: 49
  terada: 10


In [10]:
# サンプルデータ表示
samples = conn.execute("""
    SELECT id, category, file_name, width, height, format 
    FROM image_catalog 
    LIMIT 10
""").fetchall()

print("サンプルデータ:")
for row in samples:
    print(f"  {row}")

サンプルデータ:
  ('b403d4e8-9431-4762-9a2a-3d4f14e0f444', 'EuroPython2025', '2025-07-16 00.24.07.jpg', 5472, 3072, 'MPO')
  ('23a26e21-4b89-45f5-aae7-ab11083f1f08', 'EuroPython2025', '2025-07-16 00.37.53.jpg', 5472, 3072, 'MPO')
  ('b9ad5e2c-da0a-4309-903f-30bb2792467a', 'EuroPython2025', '2025-07-16 00.38.00.jpg', 5472, 3072, 'MPO')
  ('d39db8eb-d9ff-4854-886c-42c551aa146d', 'EuroPython2025', '2025-07-16 16.03.32.jpg', 6000, 4000, 'MPO')
  ('dd81e73e-ac1e-4025-b57e-e34b9f291fa9', 'EuroPython2025', '2025-07-16 16.05.21.jpg', 6000, 4000, 'MPO')
  ('d1945a5f-bfa8-4362-ab14-55de6a51a060', 'EuroPython2025', '2025-07-16 16.06.02.jpg', 6000, 4000, 'MPO')
  ('add41e3a-659b-4a10-9522-daeafda9378a', 'EuroPython2025', '2025-07-16 00.48.42.jpg', 5472, 3072, 'MPO')
  ('f81be7c3-8a7f-44b5-8313-e27c44905e7d', 'EuroPython2025', '2025-07-16 00.48.44.jpg', 5472, 3072, 'MPO')
  ('5b46d7bb-ac97-4cfd-99e9-175fdd5a4220', 'EuroPython2025', '2025-07-16 16.02.49.jpg', 6000, 4000, 'MPO')
  ('7f533d50-df9b-43a3-b83b-

In [11]:
# 接続を閉じる
conn.close()
print(f"\nDuckDBファイル保存先: {DB_PATH.resolve()}")


DuckDBファイル保存先: /home/terapyon/dev/vibe-coding/image-vector-poc/data/images.duckdb
